## Setup

Carrega a base long (já em inglês) gerada pela etapa de limpeza, a partir de `../data/interim/tol_long_diagnosticado.csv`. O resultado deste notebook é salvo em um **novo** arquivo, `../data/processed/tol_long_dificuldade.csv`, acrescido da coluna `difficulty` derivada dos textos analisados (manual SAFE_TOL_BR e paper de sequence mining). A base de entrada não é modificada.

In [ ]:
import pandas as pd

# Entrada: base long gerada pela pipeline (não é sobrescrita)
CAMINHO_ENTRADA = '../data/interim/tol_long_diagnosticado.csv'

# Saída: novo dataset com a coluna de dificuldade
CAMINHO_SAIDA = '../data/processed/tol_long_dificuldade.csv'

df = pd.read_csv(
    CAMINHO_ENTRADA,
    sep=',',
    encoding='utf-8-sig',
    low_memory=False,
)

print('Shape:', df.shape)
df.head()

## Criando métricas para avaliar planejamento e solução de problemas

De acordo com o SAFE_TOL_BR_manual_v8.pdf  "A Torre de Londres (TOL-BR) avalia as funções de planejamento e solução de problemas, consideradas funções executivas complexas."

Página 11: A versão brasileira adaptada da TOL corrige o efeito teto descrito na literatura, e apresentoa maior consistência interna do que a tradicional.

  O planejamento e a solução de problemas são avaliados da seguinte forma:

- Página 27, linhas 4–7: "informa-se ao avaliando que ele deverá realizar a tarefa com a menor quantidade possível de movimentos
  das bolas entre as hastes, sendo contado um movimento sempre que ele clicar em uma bola e colocar em qualquer haste"
- Página 29, linhas 6–7: "Para a correção, o sistema considera uma resposta correta se o avaliando fizer a tarefa com a quantidade
  mínima de movimentos."
- Página 29, linhas 7–9: "Se fizer o acerto na primeira tentativa, ganha três pontos; na segunda, dois pontos; e na terceira, um
  ponto. Se não acertar na terceira, não recebe pontuação no item."
- Página 28, linhas 4–8: "O teste é composto por 19 problemas ao todo. O avaliando terá três tentativas para resolver cada problema. [...] Dos 19 problemas, as soluções variam de três a nove movimentos"

  Em resumo: a capacidade de planejamento é inferida pela habilidade de resolver problemas espaciais com o número mínimo de passos —
  quanto mais eficiente (acerto na 1ª tentativa, em problemas mais difíceis), melhor a capacidade de planejamento.

### Análise
O problema dessa técnica é que não discrimina o avaliando pelo tempo que levou fazendo cada tarefa.

In [ ]:
# Movimentos mínimos esperados pela tabela do manual (por round/item)
mov_minimo_tabela = {
    1: 3, 2: 3, 3: 4, 4: 4, 5: 5, 6: 5, 7: 5, 8: 5, 9: 5, 10: 5,
    11: 6, 12: 6, 13: 6, 14: 6, 15: 7, 16: 7, 17: 7, 18: 8, 19: 9
}

# Filtrar apenas acertos na 1ª tentativa (score == 3)
acertos_1a = df[(df['attempt'] == 1) & (df['score'] == 3)]

# Mínimo de movimentos observado por round
mov_observado = acertos_1a.groupby('round')['n_moves'].min()

# Comparar
comparacao = pd.DataFrame({
    'min_moves_table': pd.Series(mov_minimo_tabela),
    'min_moves_observed': mov_observado
})
comparacao['matches'] = comparacao['min_moves_table'] == comparacao['min_moves_observed']
print(comparacao)

## Dificuldade categórica por round (item)

Segundo a Tabela 2 do artigo *"Sequence mining of cognitive strategies in a computerized neuropsychological task"*, cada um dos 19 itens do TOL-BR recebe uma categoria de dificuldade estimada por IRT (parâmetro theta): `Easy`, `Medium easy`, `Medium`, `Medium hard` e `Hard`.

A dificuldade categórica (IRT) **não** é sinônimo do número mínimo de movimentos: por exemplo, o item 19 exige 9 movimentos mas é classificado como `Medium easy` (theta = -0.609), enquanto o item 9 exige 5 movimentos e é `Hard` (theta = 0.963).

In [ ]:
# Categoria de dificuldade por item (Tabela 2 do paper de sequence mining, estimada por IRT)
difficulty_by_round = {
    1: 'Medium easy', 2: 'Easy',        3: 'Medium easy', 4: 'Medium hard',
    5: 'Medium',      6: 'Medium hard', 7: 'Medium hard', 8: 'Easy',
    9: 'Hard',        10: 'Medium hard', 11: 'Medium',     12: 'Medium',
    13: 'Medium hard', 14: 'Medium',     15: 'Medium',     16: 'Medium',
    17: 'Medium',     18: 'Medium',     19: 'Medium easy'
}

# Ordem lógica das categorias (do mais fácil ao mais difícil) para tipo categórico ordenado
difficulty_order = ['Easy', 'Medium easy', 'Medium', 'Medium hard', 'Hard']

# Cria coluna categórica ordenada a partir do round
df['difficulty'] = pd.Categorical(
    df['round'].map(difficulty_by_round),
    categories=difficulty_order,
    ordered=True
)

# Verificacao: distribuicao de itens por categoria
print(df[['round', 'difficulty']].drop_duplicates().sort_values('round').to_string(index=False))
print()
print(df['difficulty'].value_counts().sort_index())

## Dataset gerado: `tol_long_dificuldade.csv`

A base long (em inglês) acrescida da coluna `difficulty`.

- **Formato:** CSV, codificação `utf-8-sig`
- **Granularidade:** cada linha representa uma tentativa de um participante em um round
- **Colunas demográficas:** `id`, `sex`, `birth_date`, `age`, `education`, `state`, `application_date`
- **Colunas da tentativa:** `round`, `attempt`, `score`, `start_time`, `end_time`, `n_moves`, `moves_description`, `first_correct_click_time`, `first_wrong_click_time`
- **Coluna derivada dos textos analisados:** `difficulty` — categoria de dificuldade IRT do item (`round`), segundo a Tabela 2 do paper de sequence mining

In [ ]:
# Salvar o resultado em um novo dataset (sem sobrescrever a entrada)
from pathlib import Path

Path(CAMINHO_SAIDA).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CAMINHO_SAIDA, index=False, encoding='utf-8-sig')

print(f'Arquivo salvo: {CAMINHO_SAIDA}')
print(f'Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]}')
print('Colunas:', list(df.columns))